Importing Dependencies

In [8]:
import os
import yaml
from pathlib import Path
from tqdm import tqdm
import shutil
import yaml

Creating Path and Folders

In [9]:
Dataset_Folder = Path(os.environ["MHCD2022_PATH"])

Labels_root = Dataset_Folder.parent / "MHCD2022_YOLO_Labels"
YOLO_root = Dataset_Folder.parent / "MHCD2022_YOLO"

for split in ["train", "val", "test"]:

    (YOLO_root / "images" / split).mkdir(
        parents=True,
        exist_ok=True
    )

    (YOLO_root / "labels" / split).mkdir(
        parents=True,
        exist_ok=True
    )


Copying from the RAW Dataset Folder

In [10]:
split_dir = Dataset_Folder / "ImageSets" / "Main"

for split in ["train", "val", "test"]:

    split_file = split_dir / f"{split}.txt"

    with open(split_file) as f:
        ids = [
            line.strip()
            for line in f
            if line.strip()
        ]

    print(f"\nProcessing {split}: {len(ids)} images")

    for stem in tqdm(ids):

        img_src = (
            Dataset_Folder
            / "JPEGImages"
            / f"{stem}.jpg"
        )

        lbl_src = (
            Labels_root
            / f"{stem}.txt"
        )

        img_dst = (
            YOLO_root
            / "images"
            / split
            / f"{stem}.jpg"
        )

        lbl_dst = (
            YOLO_root
            / "labels"
            / split
            / f"{stem}.txt"
        )

        if img_src.exists():
            shutil.copy2(img_src, img_dst)

        if lbl_src.exists():
            shutil.copy2(lbl_src, lbl_dst)

print("\nDataset organization complete.")


Processing train: 2400 images


100%|██████████| 2400/2400 [00:02<00:00, 1048.53it/s]



Processing val: 480 images


100%|██████████| 480/480 [00:00<00:00, 1133.56it/s]



Processing test: 600 images


100%|██████████| 600/600 [00:00<00:00, 1100.22it/s]


Dataset organization complete.


Create data.yaml

In [11]:
yaml_data = {
    "path": str(YOLO_root),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",

    "names": {
        0: "person",
        1: "military vehicle",
        2: "tank",
        3: "aeroplane",
        4: "warship"
    }
}

with open(YOLO_root / "data.yaml", "w") as f:
    yaml.dump(
        yaml_data,
        f,
        sort_keys=False
    )

print("data.yaml created")

data.yaml created


Verifying The Created Dataset

In [12]:
for split in ["train", "val", "test"]:

    imgs = len(
        list(
            (YOLO_root / "images" / split).glob("*.jpg")
        )
    )

    labels = len(
        list(
            (YOLO_root / "labels" / split).glob("*.txt")
        )
    )

    print(
        f"{split}: {imgs} images | {labels} labels"
    )

train: 2400 images | 2400 labels
val: 480 images | 480 labels
test: 600 images | 600 labels
